# OOP Basics

Core object-oriented programming concepts in Python: classes, instantiation, `self`, `__init__`, and inheritance/extension — with comparisons to Java where useful.

In [3]:
# TODO: Implement a class `Dataset` that:
# - takes a list of items in its constructor and stores them
# - supports len(dataset) -> number of items
# - supports dataset[i] -> the i-th item


class Dataset:
    def __init__(self, lst):
        self._lst = lst
    
    def __len__(self):
        return len(self._lst)
    
    def __getitem__(self, indx):
        return self._lst[indx]

d = Dataset([1,2,3])
print(d[0])

1


## `__init__` and dunder methods — what's actually "built in"?

Not every class needs an `__init__`. If you don't define one, the class just inherits the default `__init__` from `object`, which does nothing — it creates a bare instance with no attributes set. You only need to write your own `__init__` if you want to set up instance state (attributes) when the object is created.

`__init__`, `__len__`, `__getitem__` are **dunder methods** ("double underscore"), also called magic methods. They are *not* built-in in the sense that Python writes their logic for you — you implement them yourself. What's special is that the Python interpreter has hardcoded hooks: specific syntax or built-in functions automatically look for a specifically-named method on your object and call it.

- `Dataset(...)` → Python calls `__new__` (creates the object), then `__init__` (initializes it).
- `len(d)` → Python looks for `d.__len__()` and calls it.
- `d[i]` → Python looks for `d.__getitem__(i)` and calls it.

This is how a custom class plugs into built-in syntax/functions — often called operator overloading, or implementing a "protocol." The double-underscore naming is a convention Python itself recognizes for a **fixed, specific set of names** tied to specific operations (`__init__`, `__len__`, `__getitem__`, `__add__`, `__eq__`, `__iter__`, `__str__`, `__enter__`/`__exit__`, etc.). Inventing your own, e.g. `__my_thing__`, does nothing special — it's just a regular method with an odd name; nothing calls it automatically.

In [ ]:
# TODO: Implement a class `LabeledDataset` that extends `Dataset` so that:
# - it takes the same list of items, plus a parallel list of labels
# - dataset[i] -> (item, label) instead of just item
# - len(dataset) still works as before

class Dataset:
    def __init__(self, lst):
        self._lst = lst
    
    def __len__(self):
        return len(self._lst)
    
    def __getitem__(self, indx):
        return self._lst[indx]

class LabeledDataset(Dataset):
    '''
    this class is an extension of the Dataset class
    args: item and label where 
    label[i] is the label for item[i]
    this dataset supports indexing and return the length of the dataset
    '''
    def __init__(self, item, label):
        assert len(item) == len(label), "two arrays should have the same length!!"
        super().__init__(item)
        self._label = label
        
    def __getitem__(self, indx):
        return (self._lst[indx], self._label[indx])
    
    '''
    ## another solution 
    def __getitem__(self, indx):
        return(super().__getitem__(indx), self._label[indx])
    '''

label = [1, [], 2, 4]
item = ['a', 'b', 'c', 'd']

data = LabeledDataset(item, label)
print(len(data))
print(data[1])

4
('b', [])


## Extending a class — lessons from `LabeledDataset`

- **`super()`** is a built-in function (not a dunder — there's no `__super__`) that gives you a proxy to call the parent class's methods.
- **One instance, built in two steps.** Creating a `LabeledDataset` makes a single object. The subclass's own `__init__` sets up the new state (labels); `super().__init__(...)` calls the *parent's* `__init__` on that same `self` to set up the inherited state (`self._lst`). Not two separate objects — one object, initialized in two stages.
- **Overriding doesn't erase the original.** Redefining a method in a subclass doesn't touch or delete the parent's version — it's still intact on the parent class. Python's method lookup checks the instance's actual class first, so the subclass's version is found and used first; the parent's version is still reachable via `super()`.
- **Validate before assigning.** Put input validation (e.g. `assert len(item) == len(label)`) as the *first* line of `__init__`, before any attribute assignment — including before `super().__init__(...)`. If validation fails, the object should never end up half-constructed.
- **Reuse via `super()` instead of duplicating logic.** If a subclass method extends behavior the parent already implements, prefer calling `super().method(...)` over re-implementing it directly:
  ```python
  def __getitem__(self, indx):
      return super().__getitem__(indx), self._label[indx]   # reuses Dataset's lookup
  ```
  rather than reaching into `self._lst[indx]` directly — keeps the logic in one place, so improvements to the parent automatically propagate.
- **Only override what actually changes.** `__len__` didn't need to be redefined in `LabeledDataset` — the inherited version from `Dataset` already does the right thing, since `self._lst` is set up identically via `super().__init__()`.